In [ ]:
import os
import dotenv
import pandas as pd

from zettelsortierung import DataBase
from zettelsortierung.nn.datasets.parquet_dataset import ParquetDataset
from zettelsortierung.nn.datasets.transforms import mobile_net_infer_transform
from zettelsortierung import (
    Composition,
    Classification,
    Batch,
    SequentialApp,
    Flatten,
    Zettel,
    Label,
    PredictionModel,
    MobileNetV3ModelSmall,
    Classifiers,
    Scan,
    Sammlungen,
)

dotenv.load_dotenv()

True

### Connect to Databbase

In [2]:
#connection_string = "sqlite:////home/jan/Dokumente/IT-Beratung Raring/Zettelsortierung/data/interim/pre-zettel.db"
connection_string = "sqlite:////home/jan/Dokumente/IT-Beratung Raring/Zettelsortierung/data/interim/testing.db"
db = DataBase(connection_string=connection_string)

### Load Zettel

In [3]:
root = "../data/processed/Scans.parquet"
df = pd.read_parquet(root)
hdf = df[df["path"].str.contains("1#")]
paths = list(hdf["path"])[:3000]

### Wrap List of Zettels as Dataset

In [4]:
num_classes = 433
num_epochs = 50
root = os.getenv("PROJECT_ROOT")
device = os.getenv("TORCH_DEVICE")
assert root is not None
assert device is not None

train_set = ParquetDataset(
    f"{root}/data/interim/{num_classes}_classes.parquet", train=True
)
model = MobileNetV3ModelSmall.from_pretrained(
    path_str=f"{root}/models/mobile_net_v3_small_{num_classes}_classes_{num_epochs}_epochs",
    num_classes=num_classes,
)
classes = train_set.get_classes()

In [8]:
def print_res(results):
    for classification in results:
        assert not isinstance(classification, Scan)
        print(
            f"{classification.zettel.id}  ->  {classification.label.sammlung.name}\t\t({classification.classifier.name}, {classification.label.confidence*100:.2f})"  # type: ignore
        )
    return results

In [7]:
assert device is not None
pipeline = Composition(
    Batch(1000),
    SequentialApp(
        PredictionModel(
            model=model,
            device=device,
            batch_size=10,
            paths=paths,
            classifier=Classifiers.MODEL_V20083,
            classes=classes
        ),
        print_res
        # ToDataBase(db_adder=db.merge_classifications)
    ),
    Flatten(),
)

results = pipeline(paths)  # type: ignore

100%|██████████| 100/100 [00:31<00:00,  3.16it/s]
1it [00:32, 32.26s/it]

A01-00000001  ->  ARING		(MODEL_V20083, 23.02)
A01-00000003  ->  WAL_RO		(MODEL_V20083, 87.16)
A01-00000005  ->  VORHELM		(MODEL_V20083, 7.92)
A01-00000007  ->  ISERLOHN		(MODEL_V20083, 21.76)
A01-00000009  ->  GESCHER		(MODEL_V20083, 75.80)
A01-00000011  ->  GESCHER		(MODEL_V20083, 90.38)
A01-00000013  ->  RAKERS_MB		(MODEL_V20083, 84.98)
A01-00000015  ->  KLOENTRUP		(MODEL_V20083, 68.61)
A01-00000017  ->  ANON_ALT_HA		(MODEL_V20083, 12.74)
A01-00000019  ->  KLEINN		(MODEL_V20083, 92.74)
A01-00000021  ->  MUELLER_D		(MODEL_V20083, 6.84)
A01-00000023  ->  OESTERHAUS		(MODEL_V20083, 87.33)
A01-00000025  ->  RAKERS_MB		(MODEL_V20083, 16.70)
A01-00000027  ->  BLANK		(MODEL_V20083, 42.59)
A01-00000029  ->  WARENDORF_NB		(MODEL_V20083, 13.69)
A01-00000031  ->  ALTENA		(MODEL_V20083, 20.57)
A01-00000033  ->  OESTERHAUS		(MODEL_V20083, 81.66)
A01-00000035  ->  ECKARDT		(MODEL_V20083, 59.41)
A01-00000037  ->  SCHWAGMEYER		(MODEL_V20083, 33.85)
A01-00000039  ->  OESTERHAUS		(MODEL_V20083, 78.29

100%|██████████| 100/100 [00:31<00:00,  3.15it/s]
2it [01:04, 32.24s/it]

A01-00000001  ->  ARCH_RKLH		(MODEL_V20083, 86.29)
A01-00000003  ->  ARCH_RKLH		(MODEL_V20083, 57.69)
A01-00000005  ->  MUESCHEDE		(MODEL_V20083, 65.40)
A01-00000007  ->  PHONEM		(MODEL_V20083, 61.05)
A01-00000009  ->  WESSUM		(MODEL_V20083, 7.05)
A01-00000011  ->  BEISENHERZ		(MODEL_V20083, 93.09)
A01-00000013  ->  WESSUM		(MODEL_V20083, 11.89)
A01-00000015  ->  MUESCHEDE		(MODEL_V20083, 56.35)
A01-00000017  ->  ALTENA		(MODEL_V20083, 39.85)
A01-00000019  ->  PHONEM		(MODEL_V20083, 33.73)
A01-00000021  ->  FREDERKING		(MODEL_V20083, 39.77)
A01-00000023  ->  SANDEBECK		(MODEL_V20083, 92.57)
A01-00000025  ->  LANDOIS_FE		(MODEL_V20083, 5.69)
A01-00000027  ->  RAKERS		(MODEL_V20083, 86.44)
A01-00000029  ->  KOESTER_JK		(MODEL_V20083, 92.42)
A01-00000031  ->  FRAGEBOGEN		(MODEL_V20083, 83.60)
A01-00000033  ->  OEKE		(MODEL_V20083, 64.20)
A01-00000035  ->  WOE		(MODEL_V20083, 14.08)
A01-00000037  ->  ECKARDT		(MODEL_V20083, 96.80)
A01-00000039  ->  ANON_BRI_AA		(MODEL_V20083, 86.58)
A01-00

100%|██████████| 100/100 [00:32<00:00,  3.11it/s]
3it [01:37, 32.43s/it]

A01-00000001  ->  SCHOENING		(MODEL_V20083, 23.43)
A01-00000003  ->  WAL_RO		(MODEL_V20083, 89.91)
A01-00000005  ->  ARCH_RKLH		(MODEL_V20083, 78.87)
A01-00000007  ->  FREDERKING		(MODEL_V20083, 28.73)
A01-00000009  ->  RHEINE		(MODEL_V20083, 75.83)
A01-00000011  ->  ANON_BBR_BH		(MODEL_V20083, 64.80)
A01-00000013  ->  OESTRICH_L		(MODEL_V20083, 64.38)
A01-00000015  ->  ANON_ISL_GS		(MODEL_V20083, 41.36)
A01-00000017  ->  VORHELM		(MODEL_V20083, 14.44)
A01-00000019  ->  GOEHNER		(MODEL_V20083, 77.08)
A01-00000021  ->  GOEHNER		(MODEL_V20083, 72.89)
A01-00000023  ->  ISERLOHN		(MODEL_V20083, 65.26)
A01-00000025  ->  RHEIN_FRGB		(MODEL_V20083, 49.47)
A01-00000027  ->  VORHELM		(MODEL_V20083, 23.04)
A01-00000029  ->  LUECHTRINGEN		(MODEL_V20083, 38.28)
A01-00000031  ->  DEILINGHOFEN		(MODEL_V20083, 61.74)
A01-00000033  ->  RHEINE		(MODEL_V20083, 50.98)
A01-00000035  ->  FRAGEBOGEN		(MODEL_V20083, 94.24)
A01-00000037  ->  FRAGEBOGEN		(MODEL_V20083, 95.47)
A01-00000039  ->  FRAGEBOGEN		(MOD

TypeError: 'NoneType' object is not iterable

In [ ]:
#for classification in results:
#    assert not isinstance(classification, Scan)
#    print(
#        f"{classification.zettel.id}  ->  {classification.label.sammlung.name}\t\t({classification.classifier.name}, {classification.label.confidence*100:.2f})"  # type: ignore
#    )

A01-00000001  ->  ARING		(MODEL_V20083, 23.02)
A01-00000003  ->  WAL_RO		(MODEL_V20083, 87.16)
A01-00000005  ->  VORHELM		(MODEL_V20083, 7.92)
A01-00000007  ->  ISERLOHN		(MODEL_V20083, 21.76)
A01-00000009  ->  GESCHER		(MODEL_V20083, 75.80)
A01-00000011  ->  GESCHER		(MODEL_V20083, 90.38)
A01-00000013  ->  RAKERS_MB		(MODEL_V20083, 84.98)
A01-00000015  ->  KLOENTRUP		(MODEL_V20083, 68.61)
A01-00000017  ->  ANON_ALT_HA		(MODEL_V20083, 12.74)
A01-00000019  ->  KLEINN		(MODEL_V20083, 92.74)
A01-00000021  ->  MUELLER_D		(MODEL_V20083, 6.84)
A01-00000023  ->  OESTERHAUS		(MODEL_V20083, 87.33)
A01-00000025  ->  RAKERS_MB		(MODEL_V20083, 16.70)
A01-00000027  ->  BLANK		(MODEL_V20083, 42.59)
A01-00000029  ->  WARENDORF_NB		(MODEL_V20083, 13.69)
A01-00000031  ->  ALTENA		(MODEL_V20083, 20.57)
A01-00000033  ->  OESTERHAUS		(MODEL_V20083, 81.66)
A01-00000035  ->  ECKARDT		(MODEL_V20083, 59.41)
A01-00000037  ->  SCHWAGMEYER		(MODEL_V20083, 33.85)
A01-00000039  ->  OESTERHAUS		(MODEL_V20083, 78.29